<a href="https://colab.research.google.com/github/Hema-14052005/Hema-14052005/blob/main/Fertilizer_suggestion_based_on_plant_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import zipfile
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Extract the dataset
zip_path = "/content/archive.zip"
extract_path = "/mnt/data/extracted_data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define constants
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "/mnt/data/rice_disease_model.h5"
DATASET_PATH = os.path.join(extract_path, "Rice_Diseases")

# Image Data Generator (Augmentation & Normalization)
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Define CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train model (only if not already trained)
if not os.path.exists(MODEL_PATH):
    model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)
    model.save(MODEL_PATH)
else:
    model = load_model(MODEL_PATH)
    model.evaluate(val_generator)  # Add this line to evaluate the loaded model



# Save class labels
class_labels = list(train_generator.class_indices.keys())

def preprocess_image(img_path):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0) / 255.0
    return img

# Predict disease
img_path = "/content/RiceSheathArk.jpg"  # Placeholder for an actual image
processed_img = preprocess_image(img_path)
prediction = model.predict(processed_img)
predicted_class = np.argmax(prediction)
predicted_disease = class_labels[predicted_class]
print(f"Predicted Disease: {predicted_disease}")

# Load fertilizer dataset
df = pd.read_csv("/content/corrected_plant_disease_fertilizer_dataset.csv")

# Map disease to fertilizer
fertilizer_mapping = dict(zip(df["Disease"], df["Recommended Fertilizer"]))
recommended_fertilizer = fertilizer_mapping.get(predicted_disease, "No fertilizer recommendation available.")
print(f"Recommended Fertilizer: {recommended_fertilizer}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/archive.zip'

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import zipfile
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

# Extract dataset
zip_path = "/content/archive.zip"
extract_path = "/mnt/data/extracted_data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define constants
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "/mnt/data/rice_disease_model.h5"
DATASET_PATH = os.path.join(extract_path, "Rice_Diseases")

# Image Data Generator (Augmentation & Normalization)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Define CNN model
model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),  # Explicit input layer
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

# Compile model before training or loading
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train model if not already saved
if not os.path.exists(MODEL_PATH):
    model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)
    model.save(MODEL_PATH)
else:
    model = load_model(MODEL_PATH)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.evaluate(val_generator)

# Save class labels
class_labels = list(train_generator.class_indices.keys())

def preprocess_image(img_path):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0) / 255.0
    return img

# Predict disease
img_path = "/content/RiceSheathArk.jpg"  # Placeholder for actual image
processed_img = preprocess_image(img_path)

@tf.function(reduce_retracing=True)
def predict_image(model, img):
    return model(img)

prediction = predict_image(model, processed_img)
predicted_class = np.argmax(prediction)
predicted_disease = class_labels[predicted_class]
print(f"Predicted Disease: {predicted_disease}")

# Load fertilizer dataset
df = pd.read_csv("/content/corrected_plant_disease_fertilizer_dataset.csv")

# Map disease to fertilizer
fertilizer_mapping = dict(zip(df["Disease"], df["Recommended Fertilizer"]))
recommended_fertilizer = fertilizer_mapping.get(predicted_disease, "No fertilizer recommendation available.")
print(f"Recommended Fertilizer: {recommended_fertilizer}")


Found 160 images belonging to 4 classes.
Found 40 images belonging to 4 classes.


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 135ms/step - accuracy: 0.7333 - loss: 0.6072
Predicted Disease: False Smut Disease
Recommended Fertilizer: Balanced NPK fertilizer and improve field sanitation


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import zipfile
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Extract the dataset
zip_path = "/content/archive.zip"
extract_path = "/mnt/data/extracted_data"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Define constants
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "/mnt/data/rice_disease_model.h5"
DATASET_PATH = os.path.join(extract_path, "Rice_Diseases")

# Image Data Generator (Augmentation & Normalization)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Define CNN model
model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

# Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train model if not already trained
if not os.path.exists(MODEL_PATH):
    print("Training model...")
    model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)
    model.save(MODEL_PATH)
    print("Model saved successfully.")
else:
    print("Loading existing model...")
    model = load_model(MODEL_PATH)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.evaluate(val_generator)

# Save class labels
class_labels = list(train_generator.class_indices.keys())

# Image preprocessing function
def preprocess_image(img_path):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0) / 255.0
    return img

# Predict disease
img_path = "/content/RiceSheathArk.jpg"  # Change this to your test image path
processed_img = preprocess_image(img_path)
prediction = model.predict(processed_img)
predicted_class = np.argmax(prediction)
predicted_disease = class_labels[predicted_class]
print(f"Predicted Disease: {predicted_disease}")

# Load fertilizer dataset
df = pd.read_csv("/content/corrected_plant_disease_fertilizer_dataset.csv")

# Map disease to fertilizer
fertilizer_mapping = dict(zip(df["Disease"], df["Recommended Fertilizer"]))
recommended_fertilizer = fertilizer_mapping.get(predicted_disease, "No fertilizer recommendation available.")
print(f"Recommended Fertilizer: {recommended_fertilizer}")


Found 160 images belonging to 4 classes.
Found 40 images belonging to 4 classes.
Loading existing model...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.6479 - loss: 0.6609


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Predicted Disease: False Smut Disease
Recommended Fertilizer: Balanced NPK fertilizer and improve field sanitation


In [ ]:
from IPython import get_ipython
from IPython.display import display
# %%
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import zipfile
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input  # Import Input

# Extract the dataset
zip_path = "/content/archive.zip"
extract_path = "/mnt/data/extracted_data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define constants
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "/mnt/data/rice_disease_model.h5"
DATASET_PATH = os.path.join(extract_path, "Rice_Diseases")

# Image Data Generator (Augmentation & Normalization)
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Define CNN model using Input layer
model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),  # Use Input layer for shape
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train model (only if not already trained)
if not os.path.exists(MODEL_PATH):
    model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)
    model.save(MODEL_PATH)
else:
    model = load_model(MODEL_PATH)
    model.evaluate(val_generator)  # Evaluate to build metrics

# ... (Rest of the code for prediction and fertilizer recommendation)

Found 160 images belonging to 4 classes.
Found 40 images belonging to 4 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.7167 - loss: 0.5892
